In [25]:
import sys, os, time
import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())

from text_utils import build_item_texts, build_query_texts, tokenize_all
from bm25_index import BM25Index
from ann_search import search_topk
from bert_two_tower import TwoTowerConfig, TwoTowerModel, mine_hard_negative_item_indices, train_two_tower
from fuse_and_export import (
    fuse_candidates_for_query, build_answer_dataframe,
    validate_answer_df, write_answer_csv,
)
from local_eval import make_local_split, recall_at_k, build_recall_report

DATA_DIR = "data"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Импорты ОК")

Импорты ОК


In [23]:
train_df = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
bench_q = pd.read_parquet(os.path.join(DATA_DIR, "benchmark_queries.parquet"))
bench_items = pd.read_parquet(os.path.join(DATA_DIR, "benchmark_items.parquet"))

print("train:", train_df.shape)
print("benchmark_queries:", bench_q.shape)
print("benchmark_items:", bench_items.shape)

print("\nКолонки train:", train_df.columns.tolist())
print("Колонки benchmark_queries:", bench_q.columns.tolist())
print("Колонки benchmark_items:", bench_items.columns.tolist())
print(train_df.iloc[0])

train: (497673, 19)
benchmark_queries: (2452, 6)
benchmark_items: (189212, 14)

Колонки train: ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category', 'item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']
Колонки benchmark_queries: ['query_id', 'search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
Колонки benchmark_items: ['item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']
search_query                                

In [27]:
item_texts = build_item_texts(bench_items)
query_texts = build_query_texts(bench_q)
item_ids = bench_items["item_id"].astype(str).tolist()
query_ids = bench_q["query_id"].astype(str).tolist()
item_tokens = tokenize_all(item_texts)
bm25 = BM25Index().fit(item_tokens)
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(query_texts), size=5, replace=False)
sample_tokens = [tokenize_all([query_texts[i]])[0] for i in sample_idx]
sample_results = bm25.batch_topk(sample_tokens, k=3, show_progress=False)

for i, (idx, scores) in zip(sample_idx, sample_results):
    print(f"\nЗапрос: {query_texts[i][:80]!r}")
    for rank, j in enumerate(idx):
        print(f"   {rank+1}. [{scores[rank]:.2f}] {item_texts[j][:90]!r}")


Запрос: 'баня на воде [SEP] категория 114'
   1. [19.88] 'Русская баня на воде(баня на дровах) [SEP] Предлагаем Вам посетить нашу «Русскую баню на в'
   2. [19.04] 'Баня на дровах [SEP] Баня на дровах, Дом на воде - аренда на сутки или почасовая аренда! П'
   3. [18.05] 'Аренда яхты и гидроцикла [SEP] 🌊 Идеальный отдых на воде уже ждет вас! 🛥️🔥\n\nХотите устроит'

Запрос: 'ремонт фрезерных станков [SEP] Тип услуги Другое Вид услуги Ремонт и обслуживани'
   1. [37.56] 'Ремонт лазерного оборудования [SEP] Ремонт и обслуживание китайский СО2 лазеров, таких про'
   2. [34.90] 'Ремонт токарных и фрезерных станков [SEP] Ремонт токарных фрезерных станков выполняем выез'
   3. [34.89] 'Пнр. Ремонт станков, восстановление геометрии [SEP] ПНР. Pемонт cтaнков и любого дpугого п'

Запрос: 'визитки реклама [SEP] Вид услуги [SEP] категория 114'
   1. [17.49] 'Баннеры,вывески,наклейки,визитки,листовки [SEP] Типография полного цикла АБВ полиграф прив'
   2. [16.80] 'Печать, Реклама, Баннеры, Световые

In [36]:
import importlib, local_eval
importlib.reload(local_eval)
from local_eval import make_local_split, recall_at_k, build_recall_report

eval_queries_df, corpus_item_texts, eval_query_texts, true_item_positions, corpus_df = make_local_split(
    train_df, n_eval_queries=3000, corpus_size=150_000, random_state=RANDOM_STATE
)
print(f"  локальный корпус: {len(corpus_item_texts)} объявлений, eval-запросов: {len(eval_query_texts)}")

local_bm25 = BM25Index().fit(tokenize_all(corpus_item_texts))

local_bm25_results = local_bm25.batch_topk(tokenize_all(eval_query_texts), k=50)
local_bm25_ranked = [list(idx) for idx, _ in local_bm25_results]


bm25_recall = recall_at_k(local_bm25_ranked, true_item_positions)
print(build_recall_report("BM25 only (local)", bm25_recall))

Собираю локальный мини-бенчмарк из train.parquet...
  локальный корпус: 150000 объявлений, eval-запросов: 3000
Строю BM25-индекс по локальному корпусу...
  готово за 34.6 c
Ищу top-50 для eval-запросов...


BM25 retrieval: 100%|██████████| 3000/3000 [01:45<00:00, 28.37it/s]


  готово за 105.9 c
[BM25 only (local)] локальный Recall@50 = 0.2360


In [ ]:
N_TRAIN_PAIRS = 50_000
N_HARD = 4

rng = np.random.default_rng(RANDOM_STATE)
train_sample_idx = rng.choice(len(train_df), size=min(N_TRAIN_PAIRS, len(train_df)), replace=False)
train_sample_df = train_df.iloc[train_sample_idx].reset_index(drop=True)
train_query_texts = build_query_texts(train_sample_df)
train_item_texts = build_item_texts(train_sample_df)

train_query_tokens = tokenize_all(train_query_texts)
hard_pool_results = bm25.batch_topk(train_query_tokens, k=50, show_progress=True)

hard_negative_indices = [list(idx[:N_HARD]) for idx, _ in hard_pool_results]
print("\nПример: запрос ->", train_query_texts[0][:80])
print("Позитив ->", train_item_texts[0][:80])
print("Hard negatives (тексты из корпуса benchmark_items):")
for j in hard_negative_indices[0]:
    print("  -", item_texts[j][:80])

In [ ]:
cfg = TwoTowerConfig(
    model_name="cointegrated/rubert-tiny2",
    projection_dim=128,
    max_length=64,
    temperature=0.05,
    n_hard_negatives=N_HARD,
    batch_size=32,
    epochs=3,
    lr=2e-5,
    device="cpu",
)
model = TwoTowerModel(cfg)
model = train_two_tower(
    model,
    query_texts=train_query_texts,
    item_texts_for_positive=train_item_texts,
    all_item_texts=item_texts,
    hard_negative_indices=hard_negative_indices,
    config=cfg,
)

In [28]:
item_emb = model.encode(item_texts, batch_size=128)
query_emb = model.encode(query_texts, batch_size=128)

np.save("item_embeddings.npy", item_emb)
np.save("query_embeddings.npy", query_emb)

sample_query_emb = query_emb[sample_idx]
sample_ann_results = search_topk(sample_query_emb, item_emb, k=3, use_faiss=False)
for i, (idx, scores) in zip(sample_idx, sample_ann_results):
    print(f"\nЗапрос: {query_texts[i][:80]!r}")
    for rank, j in enumerate(idx):
        print(f"   {rank+1}. [{scores[rank]:.3f}] {item_texts[j][:90]!r}")

Считаю эмбеддинги корпуса объявлений бенчмарка (189k)...


Encoding: 100%|██████████| 1479/1479 [02:25<00:00, 10.18it/s]


  готово за 145.4 c, форма: (189212, 128)
Считаю эмбеддинги запросов бенчмарка (2452)...


Encoding: 100%|██████████| 20/20 [00:00<00:00, 22.45it/s]


  готово за 0.9 c, форма: (2452, 128)
Эмбеддинги сохранены в item_embeddings.npy / query_embeddings.npy

Ищу top-3 ANN-кандидатов для тех же 5 случайных запросов, что смотрели у BM25...

Запрос: 'баня на воде [SEP] категория 114'
   1. [0.822] 'Баня на воде. Дом на воде. Хаусбот.отдых на реке [SEP] с 10 мая открытие сезона\r\nВ стоимос'
   2. [0.818] 'Баня с бассейном до 30 гостей [SEP] 🔥 Русская баня на дровах 250 м² — бассейн 20 м, караок'
   3. [0.813] 'Баня на дровах с купелью [SEP] 🪵 Русская банька на дровах с уютной зоной для отдыха. Стоим'

Запрос: 'ремонт фрезерных станков [SEP] Тип услуги Другое Вид услуги Ремонт и обслуживани'
   1. [0.877] 'Ремонт станков [SEP] Ремонт универсального металорежущего оборудования. Фрезерные ,токарны'
   2. [0.872] 'Ремонт электро-бензо инструмента [SEP] Быстро и качественно от частного мастера.\n\nРемонтир'
   3. [0.859] 'Ремонт электроинструмента [SEP] Ремонт электро инструментов,многое другое...\n\nВыполняем ре'

Запрос: 'визитки реклама [SEP

In [29]:
local_item_emb = model.encode(corpus_item_texts, batch_size=128)
local_query_emb = model.encode(eval_query_texts, batch_size=128)

local_ann_results = search_topk(local_query_emb, local_item_emb, k=50, use_faiss=False)
local_ann_ranked = [list(idx) for idx, _ in local_ann_results]

bert_only_recall = recall_at_k(local_ann_ranked, true_item_positions)
print(build_recall_report("BERT only (local)", bert_only_recall))

fused_local = [
    fuse_candidates_for_query([bm_list, ann_list], top_n=50)
    for bm_list, ann_list in zip(local_bm25_ranked, local_ann_ranked)
]
hybrid_recall = recall_at_k(fused_local, true_item_positions)
print(build_recall_report("BM25 + BERT hybrid (local)", hybrid_recall))

print(build_recall_report("BM25 only", bm25_recall))
print(build_recall_report("BERT only", bert_only_recall))
print(build_recall_report("Hybrid (RRF)", hybrid_recall))

Кодирую локальный корпус и eval-запросы (из Ячейки 4) моделью BERT...


Encoding: 100%|██████████| 24/24 [00:01<00:00, 17.37it/s]


  готово за 133.8 c
Ищу top-50 ANN-кандидатов для eval-запросов...
  готово за 0.9 c
[BERT only (local)] локальный Recall@50 = 0.1650
[BM25 + BERT hybrid (local)] локальный Recall@50 = 0.2380

--- Итог сравнения ---
[BM25 only] локальный Recall@50 = 0.2360
[BERT only] локальный Recall@50 = 0.1650
[Hybrid (RRF)] локальный Recall@50 = 0.2380


In [ ]:
cfg_more = TwoTowerConfig(
    model_name=cfg.model_name,
    projection_dim=cfg.projection_dim,
    max_length=cfg.max_length,
    temperature=cfg.temperature,
    n_hard_negatives=cfg.n_hard_negatives,
    batch_size=cfg.batch_size,
    epochs=4,
    lr=1e-5,
    device=cfg.device,
)

model = train_two_tower(
    model,
    query_texts=train_query_texts,
    item_texts_for_positive=train_item_texts,
    all_item_texts=item_texts,
    hard_negative_indices=hard_negative_indices,
    config=cfg_more,
)

In [30]:
microcat_true = eval_queries_df["item_microcat_id"].tolist()
microcat_corpus = [
    corpus_item_texts
]

In [31]:
local_item_emb_v2 = model.encode(corpus_item_texts, batch_size=128)
local_query_emb_v2 = model.encode(eval_query_texts, batch_size=128)

local_ann_results_v2 = search_topk(local_query_emb_v2, local_item_emb, k=50, use_faiss=False) \
    if False else search_topk(local_query_emb_v2, local_item_emb_v2, k=50, use_faiss=False)
local_ann_ranked_v2 = [list(idx) for idx, _ in local_ann_results_v2]

bert_only_recall_v2 = recall_at_k(local_ann_ranked_v2, true_item_positions)
print(build_recall_report("BERT only, 7 эпох (local)", bert_only_recall_v2))

fused_local_v2 = [
    fuse_candidates_for_query([bm_list, ann_list], top_n=50)
    for bm_list, ann_list in zip(local_bm25_ranked, local_ann_ranked_v2)
]
hybrid_recall_v2 = recall_at_k(fused_local_v2, true_item_positions)
print(build_recall_report("BM25 + BERT hybrid, 7 эпох (local)", hybrid_recall_v2))

print(build_recall_report("BM25 only", bm25_recall))
print(build_recall_report("BERT only, 3 эпохи", bert_only_recall))
print(build_recall_report("Hybrid, 3 эпохи", hybrid_recall))
print(build_recall_report("BERT only, 7 эпох", bert_only_recall_v2))
print(build_recall_report("Hybrid, 7 эпох", hybrid_recall_v2))

Пересчитываю эмбеддинги локального корпуса и eval-запросов дообученной моделью...


Encoding: 100%|██████████| 24/24 [00:01<00:00, 19.96it/s]


  готово за 122.2 c
[BERT only, 7 эпох (local)] локальный Recall@50 = 0.1650
[BM25 + BERT hybrid, 7 эпох (local)] локальный Recall@50 = 0.2380

--- Полное сравнение ---
[BM25 only] локальный Recall@50 = 0.2360
[BERT only, 3 эпохи] локальный Recall@50 = 0.1650
[Hybrid, 3 эпохи] локальный Recall@50 = 0.2380
[BERT only, 7 эпох] локальный Recall@50 = 0.1650
[Hybrid, 7 эпох] локальный Recall@50 = 0.2380


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n_check = 500
q_idx = rng.integers(0, len(local_query_emb_v2), size=n_check)
i_idx = rng.integers(0, len(local_item_emb_v2), size=n_check)
random_sims = np.sum(local_query_emb_v2[q_idx] * local_item_emb_v2[i_idx], axis=1)

print("Похожесть СЛУЧАЙНЫХ (нерелевантных) пар запрос-объявление:")
print(f"  среднее = {random_sims.mean():.3f}, std = {random_sims.std():.3f}, "
      f"мин = {random_sims.min():.3f}, макс = {random_sims.max():.3f}")

true_sims = []
for qi, true_pos in enumerate(true_item_positions):
    sim = np.dot(local_query_emb_v2[qi], local_item_emb_v2[true_pos])
    true_sims.append(sim)
true_sims = np.array(true_sims)
print("\nПохожесть ИСТИННЫХ пар (запрос -> его реальное объявление):")
print(f"  среднее = {true_sims.mean():.3f}, std = {true_sims.std():.3f}, "
      f"мин = {true_sims.min():.3f}, макс = {true_sims.max():.3f}")

print(f"\nРазница средних (истинные - случайные): {true_sims.mean() - random_sims.mean():.3f}")

In [32]:
import numpy as np

n_check = 300
rng = np.random.default_rng(1)
sample_eval_idx = rng.choice(len(eval_query_texts), size=n_check, replace=False)

ranks = []
for qi in sample_eval_idx:
    sims = local_item_emb_v2 @ local_query_emb_v2[qi]
    true_sim = sims[true_item_positions[qi]]
    rank = int((sims > true_sim).sum()) + 1
    ranks.append(rank)

ranks = np.array(ranks)
print("Распределение ранга истинного объявления (1 = модель поставила его на первое место):")
print(f"  медиана = {np.median(ranks):.0f}")
print(f"  среднее = {ranks.mean():.0f}")
print(f"  доля с рангом <= 50   : {(ranks <= 50).mean():.3f}")
print(f"  доля с рангом <= 200  : {(ranks <= 200).mean():.3f}")
print(f"  доля с рангом <= 1000 : {(ranks <= 1000).mean():.3f}")
print(f"  доля с рангом > 10000 : {(ranks > 10000).mean():.3f}")

Распределение ранга истинного объявления (1 = модель поставила его на первое место):
  медиана = 339
  среднее = 1306
  доля с рангом <= 50   : 0.163
  доля с рангом <= 200  : 0.363
  доля с рангом <= 1000 : 0.743
  доля с рангом > 10000 : 0.017


In [37]:
def location_ranked_candidates(query_location_id, corpus_location_ids, bm25_ranked_list, top_n=50):
    same_loc_set = {i for i, loc in enumerate(corpus_location_ids) if loc == query_location_id}
    return [i for i in bm25_ranked_list if i in same_loc_set][:top_n]

corpus_location_ids = corpus_df["item_location_id"].tolist()
eval_location_ids = eval_queries_df["search_location_id"].tolist()

local_geo_ranked = [
    location_ranked_candidates(loc, corpus_location_ids, bm_list, top_n=50)
    for loc, bm_list in zip(eval_location_ids, local_bm25_ranked)
]

fused_local_geo = [
    fuse_candidates_for_query([bm_list, ann_list, geo_list], top_n=50)
    for bm_list, ann_list, geo_list in zip(local_bm25_ranked, local_ann_ranked_v2, local_geo_ranked)
]
hybrid_geo_recall = recall_at_k(fused_local_geo, true_item_positions)
print(build_recall_report("BM25 + BERT + Geo hybrid (local)", hybrid_geo_recall))
print(f"(для сравнения) без гео: {hybrid_recall_v2:.4f}")

n_with_geo_match = sum(1 for g in local_geo_ranked if len(g) > 0)
print(f"Запросов с хотя бы одним объявлением той же локации: {n_with_geo_match}/{len(local_geo_ranked)}")

[BM25 + BERT + Geo hybrid (local)] локальный Recall@50 = 0.2800
(для сравнения) без гео: 0.2380
Запросов с хотя бы одним объявлением той же локации: 1249/3000


In [38]:
import importlib, local_eval
importlib.reload(local_eval)
from local_eval import make_local_split, recall_at_k, build_recall_report

eval_queries_df, _ct, _et, true_pos_chk, corpus_df = make_local_split(
    train_df, n_eval_queries=3000, corpus_size=150_000, random_state=RANDOM_STATE
)
assert true_pos_chk == true_item_positions, "сплит изменился -- индексы не совпадают!"
print("Сплит идентичен предыдущему, corpus_df получен:", corpus_df.shape)

print("BM25 top-1000 для eval-запросов (~2 мин)...")
t0 = time.time()
deep_results = local_bm25.batch_topk(tokenize_all(eval_query_texts), k=1000)
print(f"  готово за {time.time()-t0:.1f} c")

corpus_loc = corpus_df["item_location_id"].to_numpy()
eval_loc = eval_queries_df["search_location_id"].to_numpy()

local_geo_ranked = []
for qi, (idx, _) in enumerate(deep_results):
    same_loc = idx[corpus_loc[idx] == eval_loc[qi]]
    local_geo_ranked.append(list(same_loc[:50]))

n_with_geo = sum(1 for g in local_geo_ranked if len(g) > 0)
print(f"Запросов с гео-кандидатами: {n_with_geo}/{len(local_geo_ranked)}")
print(build_recall_report("Geo-only (BM25 + локация)", recall_at_k(local_geo_ranked, true_item_positions)))

fused_local_geo = [
    fuse_candidates_for_query([bm_list, ann_list, geo_list], top_n=50)
    for bm_list, ann_list, geo_list in zip(local_bm25_ranked, local_ann_ranked_v2, local_geo_ranked)
]
print(build_recall_report("BM25 + BERT + Geo (local)", recall_at_k(fused_local_geo, true_item_positions)))
print(build_recall_report("для сравнения: BM25 + BERT без гео", hybrid_recall_v2))

Сплит идентичен предыдущему, corpus_df получен: (150000, 19)
BM25 top-1000 для eval-запросов (~2 мин)...


BM25 retrieval: 100%|██████████| 3000/3000 [01:43<00:00, 28.92it/s]


  готово за 104.3 c
Запросов с гео-кандидатами: 2471/3000
[Geo-only (BM25 + локация)] локальный Recall@50 = 0.6080
[BM25 + BERT + Geo (local)] локальный Recall@50 = 0.6453
[для сравнения: BM25 + BERT без гео] локальный Recall@50 = 0.2380


In [39]:
def norm_q(s):
    return " ".join(str(s).lower().split())

train_q_norm = train_df["search_query"].map(norm_q)
bench_q_norm = bench_q["search_query"].map(norm_q)

train_q_set = set(train_q_norm)
in_train = bench_q_norm.isin(train_q_set)
print(f"Запросов бенчмарка, чей текст есть в train: {in_train.sum()} / {len(bench_q)} ({in_train.mean():.1%})")

train_item_set = set(train_df["item_id"].astype(str))
bench_item_in_train = bench_items["item_id"].astype(str).isin(train_item_set)
print(f"Объявлений бенчмарка, которые есть в train: {bench_item_in_train.sum()} / {len(bench_items)} ({bench_item_in_train.mean():.1%})")

bench_item_set = set(bench_items["item_id"].astype(str))
q2items = (train_df.assign(q=train_q_norm, iid=train_df["item_id"].astype(str))
                   .groupby("q")["iid"].agg(lambda s: list(set(s))))

counts_all, counts_in_corpus = [], []
for q in bench_q_norm[in_train]:
    items = q2items[q]
    counts_all.append(len(items))
    counts_in_corpus.append(sum(i in bench_item_set for i in items))

counts_all = np.array(counts_all); counts_in_corpus = np.array(counts_in_corpus)
print(f"\nДля запросов, найденных в train:")
print(f"  медиана разных выбранных объявлений на запрос: {np.median(counts_all):.0f}")
print(f"  доля запросов, у которых хотя бы одно выбранное объявление есть в корпусе бенчмарка: {(counts_in_corpus > 0).mean():.1%}")
print(f"  среднее число таких объявлений в корпусе на запрос: {counts_in_corpus.mean():.1f}")

pop = train_df["item_id"].value_counts()
print(f"\nУникальных item_id в train: {len(pop)}; медиана выборов на объявление: {pop.median():.0f}; макс: {pop.max()}")

Запросов бенчмарка, чей текст есть в train: 915 / 2452 (37.3%)
Объявлений бенчмарка, которые есть в train: 18142 / 189212 (9.6%)

Для запросов, найденных в train:
  медиана разных выбранных объявлений на запрос: 3
  доля запросов, у которых хотя бы одно выбранное объявление есть в корпусе бенчмарка: 39.6%
  среднее число таких объявлений в корпусе на запрос: 1.0

Уникальных item_id в train: 344825; медиана выборов на объявление: 1; макс: 188


In [40]:
from collections import defaultdict

match_rate = (train_df["item_location_id"] == train_df["search_location_id"]).mean()
print(f"Совпадение item_location_id == search_location_id в train: {match_rate:.1%}")

corpus_loc = corpus_df["item_location_id"].to_numpy()
loc_to_idx = defaultdict(list)
for i, loc in enumerate(corpus_loc):
    loc_to_idx[loc].append(i)
loc_to_idx = {k: np.array(v) for k, v in loc_to_idx.items()}

eval_loc = eval_queries_df["search_location_id"].to_numpy()
eval_tokens = tokenize_all(eval_query_texts)

def topk_in(idxs, scores, k):
    k = min(k, len(idxs))
    part = np.argpartition(-scores, k - 1)[:k]
    return list(idxs[part[np.argsort(-scores[part])]])

within_loc_ranked = []
n_fallback = 0
t0 = time.time()
for qi in range(len(eval_query_texts)):
    idxs = loc_to_idx.get(eval_loc[qi])
    if idxs is None or len(idxs) == 0:
        within_loc_ranked.append(fuse_candidates_for_query(
            [local_bm25_ranked[qi], local_ann_ranked_v2[qi]], top_n=50))
        n_fallback += 1
        continue
    bm_scores = local_bm25.score(eval_tokens[qi])[idxs]
    ann_scores = local_item_emb_v2[idxs] @ local_query_emb_v2[qi]
    bm_list = topk_in(idxs, bm_scores, 100)
    ann_list = topk_in(idxs, ann_scores, 100)
    fused = fuse_candidates_for_query([bm_list, ann_list], top_n=50)
    if len(fused) < 50:
        # локация маленькая: добираем глобальными кандидатами до 50
        extra = [i for i in fuse_candidates_for_query(
            [local_bm25_ranked[qi], local_ann_ranked_v2[qi]], top_n=50) if i not in set(fused)]
        fused = (fused + extra)[:50]
    within_loc_ranked.append(fused)
print(f"  готово за {time.time()-t0:.1f} c, запросов без локации в корпусе: {n_fallback}")

print(build_recall_report("Поиск внутри локации: BM25+BERT", recall_at_k(within_loc_ranked, true_item_positions)))
print(build_recall_report("для сравнения: BM25+BERT+Geo (ячейка 13)", recall_at_k(fused_local_geo, true_item_positions)))

Совпадение item_location_id == search_location_id в train: 83.1%
  готово за 94.8 c, запросов без локации в корпусе: 334
[Поиск внутри локации: BM25+BERT] локальный Recall@50 = 0.8307
[для сравнения: BM25+BERT+Geo (ячейка 13)] локальный Recall@50 = 0.6453


In [41]:
bq_loc = bench_q["search_location_id"]
bi_loc_counts = bench_items["item_location_id"].value_counts()

n_in_loc = bq_loc.map(bi_loc_counts).fillna(0)
print(f"Запросов с ≥1 объявлением той же локации: {(n_in_loc > 0).mean():.1%}")
print(f"Запросов с ≥50 объявлений той же локации: {(n_in_loc >= 50).mean():.1%}")
print(f"Медиана размера локации для запроса: {n_in_loc.median():.0f}")

tr_loc = train_df["search_location_id"]
tr_counts = pd.Series(bench_items["item_location_id"]).value_counts()
print(f"\nУникальных локаций: запросы бенчмарка {bq_loc.nunique()}, объявления бенчмарка {bench_items['item_location_id'].nunique()}")
print(f"Совпадение search_location_id == item_location_id в train: {(train_df['item_location_id'] == train_df['search_location_id']).mean():.1%}")

Запросов с ≥1 объявлением той же локации: 82.6%
Запросов с ≥50 объявлений той же локации: 82.3%
Медиана размера локации для запроса: 1638

Уникальных локаций: запросы бенчмарка 273, объявления бенчмарка 2877
Совпадение search_location_id == item_location_id в train: 83.1%


In [42]:
g = train_df.assign(match=(train_df["item_location_id"] == train_df["search_location_id"])) \
            .groupby("search_is_delivery_search")["match"].agg(["mean", "size"])
print("Совпадение локаций по флагу доставки в train:\n", g, "\n")

global_ranked = [
    fuse_candidates_for_query([local_bm25_ranked[qi], local_ann_ranked_v2[qi]], top_n=50)
    for qi in range(len(eval_query_texts))
]

def mix(loc_list, glob_list, n_loc):
    out = list(loc_list[:n_loc])
    seen = set(out)
    for i in glob_list:
        if len(out) >= 50:
            break
        if i not in seen:
            out.append(i); seen.add(i)
    return out

for n_loc in [50, 45, 40, 35, 30, 25, 20]:
    mixed = [mix(within_loc_ranked[qi], global_ranked[qi], n_loc) for qi in range(len(within_loc_ranked))]
    print(build_recall_report(f"{n_loc} из локации + {50-n_loc} глобальных", recall_at_k(mixed, true_item_positions)))

is_del = eval_queries_df["search_is_delivery_search"].to_numpy() == 1
print(f"\nДоля eval-запросов с доставкой: {is_del.mean():.1%}")
for name, mask in [("с доставкой", is_del), ("без доставки", ~is_del)]:
    if mask.sum() > 0:
        r = recall_at_k([within_loc_ranked[i] for i in np.where(mask)[0]],
                        [true_item_positions[i] for i in np.where(mask)[0]])
        print(f"  Recall@50 внутри локации, {name}: {r:.4f}")

Совпадение локаций по флагу доставки в train:
                                mean    size
search_is_delivery_search                  
0                          0.831028  497661
1                          0.500000      12 

[50 из локации + 0 глобальных] локальный Recall@50 = 0.8307
[45 из локации + 5 глобальных] локальный Recall@50 = 0.8297
[40 из локации + 10 глобальных] локальный Recall@50 = 0.8237
[35 из локации + 15 глобальных] локальный Recall@50 = 0.8183
[30 из локации + 20 глобальных] локальный Recall@50 = 0.8090
[25 из локации + 25 глобальных] локальный Recall@50 = 0.7993
[20 из локации + 30 глобальных] локальный Recall@50 = 0.7847

Доля eval-запросов с доставкой: 0.0%
  Recall@50 внутри локации, без доставки: 0.8307


In [43]:
import torch
torch.save({"backbone": model.backbone.state_dict(),
            "projection": model.projection.state_dict()}, "two_tower_7ep.pt")
print("модель сохранена")

модель сохранена


In [44]:
from tqdm import tqdm

item_emb_v2 = model.encode(item_texts, batch_size=128)
query_emb_v2 = model.encode(query_texts, batch_size=128)
np.save("item_embeddings_7ep.npy", item_emb_v2)
np.save("query_embeddings_7ep.npy", query_emb_v2)

bench_item_loc = bench_items["item_location_id"].to_numpy()
bench_q_loc = bench_q["search_location_id"].to_numpy()
loc_index = {}
for i, loc in enumerate(bench_item_loc):
    loc_index.setdefault(loc, []).append(i)
loc_index = {k: np.array(v) for k, v in loc_index.items()}

all_idx = np.arange(len(item_texts))
bench_tokens = tokenize_all(query_texts)

def topk_in(idxs, scores, k):
    k = min(k, len(idxs))
    part = np.argpartition(-scores, k - 1)[:k]
    return list(idxs[part[np.argsort(-scores[part])]])

final = []
n_no_loc, n_filled = 0, 0
for qi in tqdm(range(len(query_ids)), desc="Кандидаты"):
    bm_all = bm25.score(bench_tokens[qi])
    ann_all = item_emb_v2 @ query_emb_v2[qi]
    idxs = loc_index.get(bench_q_loc[qi])

    result = []
    if idxs is not None and len(idxs) > 0:
        result = fuse_candidates_for_query(
            [topk_in(idxs, bm_all[idxs], 100), topk_in(idxs, ann_all[idxs], 100)], top_n=50)
    else:
        n_no_loc += 1

    if len(result) < 50:
        n_filled += 1
        glob = fuse_candidates_for_query(
            [topk_in(all_idx, bm_all, 100), topk_in(all_idx, ann_all, 100)], top_n=100)
        seen = set(result)
        for i in glob:
            if len(result) >= 50:
                break
            if i not in seen:
                result.append(i); seen.add(i)
    final.append(result)

print(f"Запросов без объявлений своей локации: {n_no_loc}; с добором глобальными: {n_filled}")

answer_df = build_answer_dataframe(query_ids, final, item_ids)
problems = validate_answer_df(answer_df, query_ids, item_ids, max_items_per_row=50)
print("Проблемы формата:", problems if problems else "нет")
write_answer_csv(answer_df, "answer.csv")
print("answer.csv записан:", answer_df.shape)

Эмбеддинги корпуса и запросов бенчмарка (~3 мин)...


Кандидаты: 100%|██████████| 2452/2452 [01:52<00:00, 21.85it/s]


Запросов без объявлений своей локации: 427; с добором глобальными: 435
Проблемы формата: нет
answer.csv записан: (2452, 2)


In [45]:
coords = pd.concat([
    train_df[["item_location_id", "item_latitude", "item_longitude"]],
    bench_items[["item_location_id", "item_latitude", "item_longitude"]],
])
coords["item_latitude"] = pd.to_numeric(coords["item_latitude"], errors="coerce")
coords["item_longitude"] = pd.to_numeric(coords["item_longitude"], errors="coerce")
loc_centers = coords.groupby("item_location_id")[["item_latitude", "item_longitude"]].mean()
print("Локаций с известным центром:", len(loc_centers))

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

has_center = bench_q["search_location_id"].isin(loc_centers.index)
print(f"Запросов бенчмарка с известным центром локации: {has_center.mean():.1%}")

tr = train_df[["search_location_id", "item_location_id", "item_latitude", "item_longitude"]].copy()
tr["item_latitude"] = pd.to_numeric(tr["item_latitude"], errors="coerce")
tr["item_longitude"] = pd.to_numeric(tr["item_longitude"], errors="coerce")
tr = tr.join(loc_centers.rename(columns={"item_latitude": "c_lat", "item_longitude": "c_lon"}),
             on="search_location_id").dropna()
tr["dist_km"] = haversine_km(tr["c_lat"], tr["c_lon"], tr["item_latitude"], tr["item_longitude"])
tr["same_loc"] = tr["search_location_id"] == tr["item_location_id"]

qs = [0.5, 0.75, 0.9, 0.95, 0.99]
print("\nРасстояние (км) от центра локации запроса до выбранного объявления:")
print("  та же локация     :", tr.loc[tr.same_loc, "dist_km"].quantile(qs).round(1).to_dict())
print("  другая локация    :", tr.loc[~tr.same_loc, "dist_km"].quantile(qs).round(1).to_dict())
for r in [25, 50, 100, 200, 500]:
    print(f"  доля выбранных ближе {r:>3} км от центра: {(tr.dist_km <= r).mean():.1%}")

Локаций с известным центром: 3196
Запросов бенчмарка с известным центром локации: 82.7%

Расстояние (км) от центра локации запроса до выбранного объявления:
  та же локация     : {0.5: 3.2, 0.75: 6.2, 0.9: 10.5, 0.95: 14.3, 0.99: 20.8}
  другая локация    : {0.5: 33.9, 0.75: 99.2, 0.9: 486.0, 0.95: 1151.6, 0.99: 3191.9}
  доля выбранных ближе  25 км от центра: 95.5%
  доля выбранных ближе  50 км от центра: 97.3%
  доля выбранных ближе 100 км от центра: 98.3%
  доля выбранных ближе 200 км от центра: 98.9%
  доля выбранных ближе 500 км от центра: 99.3%


In [46]:
t2 = train_df[["search_location_id", "item_latitude", "item_longitude"]].copy()
t2["item_latitude"] = pd.to_numeric(t2["item_latitude"], errors="coerce")
t2["item_longitude"] = pd.to_numeric(t2["item_longitude"], errors="coerce")
q_centers = t2.groupby("search_location_id")[["item_latitude", "item_longitude"]].median()
print("Локаций с центром по запросам train:", len(q_centers))

bench_have_loc = np.array([loc in loc_index for loc in bench_q_loc])
bench_miss = ~bench_have_loc
cov = bench_q["search_location_id"][bench_miss].isin(q_centers.index)
print(f"Бенчмарк: запросов без своей локации {bench_miss.sum()}, из них центр известен у {cov.sum()} ({cov.mean():.1%})")

corp_lat = pd.to_numeric(corpus_df["item_latitude"], errors="coerce").to_numpy()
corp_lon = pd.to_numeric(corpus_df["item_longitude"], errors="coerce").to_numpy()
miss = [qi for qi in range(len(eval_loc)) if eval_loc[qi] not in loc_to_idx]
print(f"Локально запросов без своей локации: {len(miss)}")

cache = {}
for qi in tqdm(miss, desc="скоры"):
    cache[qi] = (local_bm25.score(eval_tokens[qi]), local_item_emb_v2 @ local_query_emb_v2[qi])

def geo_candidates(qi, radius_km, min_pool=200):
    if eval_loc[qi] not in q_centers.index:
        return None
    c_lat, c_lon = q_centers.loc[eval_loc[qi]]
    d = haversine_km(c_lat, c_lon, corp_lat, corp_lon)
    d = np.where(np.isnan(d), np.inf, d)
    idxs = np.where(d <= radius_km)[0]
    if len(idxs) < min_pool:
        idxs = np.argsort(d)[:min_pool]
    bm, ann = cache[qi]
    return fuse_candidates_for_query([topk_in(idxs, bm[idxs], 100), topk_in(idxs, ann[idxs], 100)], top_n=50)

truth = [true_item_positions[qi] for qi in miss]
print(build_recall_report("Только глобально (как в боевом файле)", recall_at_k([global_ranked[qi] for qi in miss], truth)))
for R in [30, 100, 300, 1000]:
    preds, n_none = [], 0
    for qi in miss:
        g = geo_candidates(qi, R)
        if g is None:
            g, n_none = global_ranked[qi], n_none + 1
        preds.append(g)
    print(build_recall_report(f"Радиус {R} км (без центра: {n_none})", recall_at_k(preds, truth)))

Локаций с центром по запросам train: 2782
Бенчмарк: запросов без своей локации 427, из них центр известен у 427 (100.0%)
Локально запросов без своей локации: 334


скоры: 100%|██████████| 334/334 [00:11<00:00, 28.20it/s]


[Только глобально (как в боевом файле)] локальный Recall@50 = 0.3443
[Радиус 30 км (без центра: 0)] локальный Recall@50 = 0.6467
[Радиус 100 км (без центра: 0)] локальный Recall@50 = 0.7156
[Радиус 300 км (без центра: 0)] локальный Recall@50 = 0.6737
[Радиус 1000 км (без центра: 0)] локальный Recall@50 = 0.4910


In [47]:
R_KM = 100
bench_lat = pd.to_numeric(bench_items["item_latitude"], errors="coerce").to_numpy()
bench_lon = pd.to_numeric(bench_items["item_longitude"], errors="coerce").to_numpy()

final_v2 = [list(x) for x in final]
n_changed = 0
for qi in tqdm(np.where(bench_miss)[0], desc="Гео-радиус для запросов без своей локации"):
    loc = bench_q_loc[qi]
    if loc not in q_centers.index:
        continue
    c_lat, c_lon = q_centers.loc[loc]
    d = haversine_km(c_lat, c_lon, bench_lat, bench_lon)
    d = np.where(np.isnan(d), np.inf, d)
    idxs = np.where(d <= R_KM)[0]
    if len(idxs) < 200:
        idxs = np.argsort(d)[:200]
    bm_all = bm25.score(bench_tokens[qi])
    ann_all = item_emb_v2 @ query_emb_v2[qi]
    final_v2[qi] = fuse_candidates_for_query(
        [topk_in(idxs, bm_all[idxs], 100), topk_in(idxs, ann_all[idxs], 100)], top_n=50)
    n_changed += 1

print("Заменено запросов:", n_changed)
answer_v2 = build_answer_dataframe(query_ids, final_v2, item_ids)
problems = validate_answer_df(answer_v2, query_ids, item_ids, max_items_per_row=50)
print("Проблемы формата:", problems if problems else "нет")
write_answer_csv(answer_v2, "answer_v2.csv")
print("answer_v2.csv записан:", answer_v2.shape)

Гео-радиус для запросов без своей локации: 100%|██████████| 427/427 [00:20<00:00, 20.44it/s]


Заменено запросов: 427
Проблемы формата: нет
answer_v2.csv записан: (2452, 2)


In [48]:
def fuse_in(idxs, bm, ann, n=50):
    return fuse_candidates_for_query(
        [topk_in(idxs, bm[idxs], 100), topk_in(idxs, ann[idxs], 100)], top_n=n)

own_queries = [qi for qi in range(len(eval_loc))
               if eval_loc[qi] in loc_to_idx and eval_loc[qi] in q_centers.index][:1500]
print("Запросов со своей локацией в выборке:", len(own_queries))

names = ["own_loc", "own40+radius100", "own45+radius100", "radius10", "radius20", "radius50"]
hits = {n: [] for n in names}
same_loc_true = []

for qi in tqdm(own_queries, desc="стратегии"):
    bm = local_bm25.score(eval_tokens[qi])
    ann = local_item_emb_v2 @ local_query_emb_v2[qi]
    own = loc_to_idx[eval_loc[qi]]
    tp = true_item_positions[qi]
    same_loc_true.append(corpus_loc[tp] == eval_loc[qi])

    c_lat, c_lon = q_centers.loc[eval_loc[qi]]
    d = haversine_km(c_lat, c_lon, corp_lat, corp_lon)
    d = np.where(np.isnan(d), np.inf, d)

    res = {}
    res["own_loc"] = fuse_in(own, bm, ann, 50)
    if len(own) < 50:
        res["own_loc"] = mix(res["own_loc"], global_ranked[qi], 50)

    rad100 = np.where((d <= 100) & (corpus_loc != eval_loc[qi]))[0]
    for n_own, key in [(40, "own40+radius100"), (45, "own45+radius100")]:
        base = list(res["own_loc"][:n_own])
        extra = fuse_in(rad100, bm, ann, 50) if len(rad100) else []
        res[key] = mix(base, extra, n_own)

    for R, key in [(10, "radius10"), (20, "radius20"), (50, "radius50")]:
        idxs = np.where(d <= R)[0]
        if len(idxs) < 200:
            idxs = np.argsort(d)[:200]
        res[key] = fuse_in(idxs, bm, ann, 50)

    for n in names:
        hits[n].append(tp in set(res[n]))

same_loc_true = np.array(same_loc_true)

print("\nRecall@50 (по запросам со своей локацией):")
for n in names:
    h = np.array(hits[n])
    print(f"  {n:18s} всего {h.mean():.4f} | нужное в своей локации {h[same_loc_true].mean():.4f} | в другой {h[~same_loc_true].mean():.4f}")

Запросов со своей локацией в выборке: 1500


стратегии: 100%|██████████| 1500/1500 [01:02<00:00, 23.98it/s]


Доля запросов, где нужное объявление в СВОЕЙ локации: 94.7%

Recall@50 (по запросам со своей локацией):
  own_loc            всего 0.9020 | нужное в своей локации 0.9514 | в другой 0.0250
  own40+radius100    всего 0.9053 | нужное в своей локации 0.9366 | в другой 0.3500
  own45+radius100    всего 0.9093 | нужное в своей локации 0.9458 | в другой 0.2625
  radius10           всего 0.8480 | нужное в своей локации 0.8803 | в другой 0.2750
  radius20           всего 0.9087 | нужное в своей локации 0.9373 | в другой 0.4000
  radius50           всего 0.9000 | нужное в своей локации 0.9211 | в другой 0.5250


In [49]:
rating = pd.to_numeric(corpus_df["item_rating"], errors="coerce").fillna(0).to_numpy()
reviews = pd.to_numeric(corpus_df["item_rating_reviews_count"], errors="coerce").fillna(0).to_numpy()
log_rev = np.log1p(reviews)


q_sample = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
pct_rating, pct_reviews = [], []
for qi in q_sample:
    own = loc_to_idx[eval_loc[qi]]
    tp = true_item_positions[qi]
    if len(own) < 50:
        continue
    pct_rating.append((rating[own] < rating[tp]).mean())
    pct_reviews.append((reviews[own] < reviews[tp]).mean())
print(f"Перцентиль рейтинга выбранного объявления среди объявлений своей локации: медиана {np.median(pct_rating):.2f}, среднее {np.mean(pct_rating):.2f}")
print(f"Перцентиль числа отзывов выбранного объявления: медиана {np.median(pct_reviews):.2f}, среднее {np.mean(pct_reviews):.2f}")
print("(0.5 = выбор не зависит от признака; выше = выбирают с более высоким значением)\n")

# --- взвешенный RRF ---
def weighted_rrf(lists_weights, top_n=50, k=60):
    sc = {}
    for lst, w in lists_weights:
        for r, i in enumerate(lst):
            sc[i] = sc.get(i, 0.0) + w / (k + r + 1)
    return [i for i, _ in sorted(sc.items(), key=lambda kv: -kv[1])[:top_n]]

priors = {
    "rating": rating,
    "reviews": log_rev,
    "rating*log(reviews)": rating * log_rev,
}
weights = [0.3, 0.6, 1.0]
POOL = 300

variants = ["base"] + [f"{p} w={w}" for p in priors for w in weights]
hits = {v: [] for v in variants}

for qi in tqdm(q_sample, desc="варианты"):
    own = loc_to_idx[eval_loc[qi]]
    if len(own) < 100:
        continue
    bm = local_bm25.score(eval_tokens[qi])[own]
    ann = local_item_emb_v2[own] @ local_query_emb_v2[qi]
    bm_l = topk_in(own, bm, POOL)
    ann_l = topk_in(own, ann, POOL)
    tp = true_item_positions[qi]

    hits["base"].append(tp in set(weighted_rrf([(bm_l, 1.0), (ann_l, 1.0)])))

    pool = np.array(list(dict.fromkeys(bm_l + ann_l)))
    for pname, pv in priors.items():
        prior_list = list(pool[np.argsort(-pv[pool])][:100])
        for w in weights:
            res = weighted_rrf([(bm_l, 1.0), (ann_l, 1.0), (prior_list, w)])
            hits[f"{pname} w={w}"].append(tp in set(res))

print(f"Запросов в оценке: {len(hits['base'])}")
for v in variants:
    print(f"  {v:28s} Recall@50 = {np.mean(hits[v]):.4f}")

Перцентиль рейтинга выбранного объявления среди объявлений своей локации: медиана 0.45, среднее 0.38
Перцентиль числа отзывов выбранного объявления: медиана 0.49, среднее 0.49
(0.5 = выбор не зависит от признака; выше = выбирают с более высоким значением)



варианты: 100%|██████████| 1500/1500 [00:45<00:00, 33.09it/s]

Запросов в оценке: 1289
  base                         Recall@50 = 0.9123
  rating w=0.3                 Recall@50 = 0.9116
  rating w=0.6                 Recall@50 = 0.9069
  rating w=1.0                 Recall@50 = 0.8836
  reviews w=0.3                Recall@50 = 0.9108
  reviews w=0.6                Recall@50 = 0.9069
  reviews w=1.0                Recall@50 = 0.8813
  rating*log(reviews) w=0.3    Recall@50 = 0.9108
  rating*log(reviews) w=0.6    Recall@50 = 0.9061
  rating*log(reviews) w=1.0    Recall@50 = 0.8813


In [50]:
def to_bool(s):
    return s.map(lambda x: bool(x) if pd.notna(x) else False)

tr_ph = to_bool(train_df["item_is_phone_hidden"]);  tr_ms = to_bool(train_df["item_is_message_forbidden"])
bi_ph = to_bool(bench_items["item_is_phone_hidden"]); bi_ms = to_bool(bench_items["item_is_message_forbidden"])

c_ph = to_bool(corpus_df["item_is_phone_hidden"]).to_numpy()
c_ms = to_bool(corpus_df["item_is_message_forbidden"]).to_numpy()
masks = {
    "без фильтра": np.zeros(len(corpus_df), dtype=bool),
    "убрать: phone_hidden": c_ph,
    "убрать: message_forbidden": c_ms,
    "убрать: любой из двух": c_ph | c_ms,
    "убрать: оба сразу": c_ph & c_ms,
}

qs_test = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
hits = {k: [] for k in masks}
ceil = {k: [] for k in masks}
for qi in tqdm(qs_test, desc="фильтры"):
    own = loc_to_idx[eval_loc[qi]]
    bm = local_bm25.score(eval_tokens[qi])
    ann = local_item_emb_v2 @ local_query_emb_v2[qi]
    tp = true_item_positions[qi]
    for name, m in masks.items():
        keep = own[~m[own]]
        if len(keep) < 50:
            keep = own
        res = fuse_candidates_for_query(
            [topk_in(keep, bm[keep], 100), topk_in(keep, ann[keep], 100)], top_n=50)
        hits[name].append(tp in set(res))
        ceil[name].append(tp in set(keep))

print("\nRecall@50 внутри своей локации:")
for name in masks:
    print(f"  {name:28s} {np.mean(hits[name]):.4f} | выбранное осталось в пуле: {np.mean(ceil[name]):.1%}")

Доли флагов: выбранные в train | корпус бенчмарка
  телефон скрыт        : 12.1% | 15.6%
  сообщения запрещены  : 2.1% | 2.7%
  хотя бы один из двух : 14.2% | 18.3%
  оба сразу            : 0.0% | 0.0%


фильтры: 100%|██████████| 1500/1500 [00:54<00:00, 27.73it/s]


Recall@50 внутри своей локации:
  без фильтра                  0.9007 | выбранное осталось в пуле: 94.7%
  убрать: phone_hidden         0.8080 | выбранное осталось в пуле: 83.9%
  убрать: message_forbidden    0.8800 | выбранное осталось в пуле: 92.7%
  убрать: любой из двух        0.7893 | выбранное осталось в пуле: 81.9%
  убрать: оба сразу            0.9007 | выбранное осталось в пуле: 94.7%


In [51]:
def profile(df, name):
    print(f"\n===== {name}: {df.shape} =====")
    rows = []
    for c in df.columns:
        s = df[c]
        empty = s.isna() | (s.astype(str).str.strip() == "")
        top = s.value_counts(dropna=True).head(3)
        rows.append({
            "колонка": c,
            "тип": str(s.dtype),
            "пусто %": round(empty.mean() * 100, 1),
            "уникальных": s.nunique(),
            "топ значения": "; ".join(f"{str(k)[:25]}({v})" for k, v in top.items()),
        })
    print(pd.DataFrame(rows).to_string(index=False))

profile(train_df, "train")
profile(bench_q, "benchmark_queries")
profile(bench_items, "benchmark_items")

# фильтры запроса: какие бывают и как часто
print("\nТоп значений search_infm_params_text в бенчмарке:")
print(bench_q["search_infm_params_text"].value_counts().head(15))

# рейтинговый фильтр: выполняется ли он на выбранных объявлениях в train
f = train_df["search_infm_params_text"].astype(str).str.contains("Рейтинг", na=False)
print(f"\nСтрок train с рейтинговым фильтром: {f.sum()}")
if f.sum() > 0:
    r = pd.to_numeric(train_df.loc[f, "item_rating"], errors="coerce")
    print(train_df.loc[f, "search_infm_params_text"].value_counts().head(5))
    print("Распределение item_rating у выбранных:", r.describe().round(2).to_dict())

# категория запроса и объявления
print(f"\nsearch_category == item_category_id в train: {(train_df['search_category'] == train_df['item_category_id']).mean():.1%}")
print("Категории в train:", train_df["item_category_id"].nunique(), "| в корпусе бенчмарка:", bench_items["item_category_id"].nunique())

# скрытый телефон и запрет сообщений: выбранные против всего корпуса
for c in ["item_is_phone_hidden", "item_is_message_forbidden"]:
    print(f"{c}: выбранные в train {train_df[c].mean():.1%} | корпус бенчмарка {bench_items[c].mean():.1%}")

# цена
p_tr = pd.to_numeric(train_df["item_price"], errors="coerce")
p_bi = pd.to_numeric(bench_items["item_price"], errors="coerce")
print("\nцена train:", p_tr.describe().round(1).to_dict())
print("цена корпус:", p_bi.describe().round(1).to_dict())
print("доля цены == 1.0: train", f"{(p_tr == 1.0).mean():.1%}", "| корпус", f"{(p_bi == 1.0).mean():.1%}")


===== train: (497673, 19) =====
                  колонка     тип  пусто %  уникальных                                                                                    топ значения
             search_query     str      0.0       74529                                           маникюр(6540); массаж(5702); наращивание ресниц(5395)
       search_location_id   int64      0.0        2782                                                     637640(36276); 653240(25229); 107620(12277)
search_is_delivery_search   int32      0.0           2                                                                                0(497661); 1(12)
  search_infm_params_text     str     33.0        3724                    (163999); Вид услуги Автосервис, ар(19654); Вид услуги Ремонт и отдел(15586)
          search_category   int64      0.0           4                                                                       114(497635); 0(34); 81(2)
           item_title_raw     str      0.0      210029       

In [52]:
from tokenize import tokenize as _unused  # noqa (чтобы не путать с tokenize)
from text_utils import tokenize

titles = corpus_df["item_title_raw"].fillna("").astype(str).tolist()
descs  = corpus_df["item_description_raw"].fillna("").astype(str).tolist()
params = corpus_df["item_infm_params_text"].fillna("").astype(str).tolist()

def build_tokens(title_w, desc_max, params_mode):
    docs = []
    for t, d, p in zip(titles, descs, params):
        tt = tokenize(t)
        dd = tokenize(d)[:desc_max] if desc_max else []
        if params_mode == "none":
            pp = []
        elif params_mode == "dedupe":
            pp = list(dict.fromkeys(tokenize(p)))
        else:
            pp = tokenize(p)
        docs.append(tt * title_w + dd + pp)
    return docs

variants = {
    "A: заголовок x3 + описание 150 + параметры без повторов": (3, 150, "dedupe"),
    "B: заголовок x3 + описание 150, без параметров":          (3, 150, "none"),
    "C: заголовок x5 + описание 80 + параметры без повторов":  (5, 80, "dedupe"),
}

qs_test = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]

def eval_bm25(bm, q_tokens):
    hits = []
    for qi in qs_test:
        own = loc_to_idx[eval_loc[qi]]
        b = bm.score(q_tokens[qi])
        a = local_item_emb_v2 @ local_query_emb_v2[qi]
        res = fuse_candidates_for_query(
            [topk_in(own, b[own], 100), topk_in(own, a[own], 100)], top_n=50)
        hits.append(true_item_positions[qi] in set(res))
    return np.mean(hits)

# запросы: только текст search_query (без "категория 114" и фильтров) против полного текста
q_full = eval_tokens
q_short = [tokenize(x) for x in eval_queries_df["search_query"].astype(str)]

print(f"База (текущий BM25, полный текст запроса): {np.mean([hits_ for hits_ in [0]]) if False else ''}", end="")
base_hits = []
for qi in qs_test:
    own = loc_to_idx[eval_loc[qi]]
    b = local_bm25.score(q_full[qi]); a = local_item_emb_v2 @ local_query_emb_v2[qi]
    res = fuse_candidates_for_query([topk_in(own, b[own], 100), topk_in(own, a[own], 100)], top_n=50)
    base_hits.append(true_item_positions[qi] in set(res))
print(f"{np.mean(base_hits):.4f}")

for name, (tw, dm, pm) in variants.items():
    t0 = time.time()
    bm_new = BM25Index().fit(build_tokens(tw, dm, pm))
    r_full = eval_bm25(bm_new, q_full)
    r_short = eval_bm25(bm_new, q_short)
    print(f"{name}\n   запрос полный: {r_full:.4f} | только search_query: {r_short:.4f}  ({time.time()-t0:.0f} c)")

База (текущий BM25, полный текст запроса): 0.9007
A: заголовок x3 + описание 150 + параметры без повторов
   запрос полный: 0.8960 | только search_query: 0.8893  (107 c)
B: заголовок x3 + описание 150, без параметров
   запрос полный: 0.8853 | только search_query: 0.8807  (143 c)
C: заголовок x5 + описание 80 + параметры без повторов
   запрос полный: 0.8993 | только search_query: 0.8867  (85 c)


In [53]:
from text_utils import tokenize

# --- 1) запросы с категорией 0 ---
q0 = bench_q["search_category"] == 0
print("Запросов с категорией 0:", q0.sum())
print("Примеры текстов:")
for t in bench_q.loc[q0, "search_query"].sample(20, random_state=0):
    print("  -", t)

print("\nКатегории объявлений в корпусе:")
print(bench_items["item_category_id"].value_counts().head(12))

# что мы вернули этим запросам: доля кандидатов вне категории 114
item_cat = bench_items["item_category_id"].to_numpy()
def share_non114(qi):
    return (item_cat[final_v2[qi]] != 114).mean()
sh0 = np.mean([share_non114(qi) for qi in np.where(q0)[0]])
sh1 = np.mean([share_non114(qi) for qi in np.where(~q0)[0]])
print(f"\nДоля кандидатов вне категории 114 в нашем ответе: запросы cat=0: {sh0:.1%} | cat=114: {sh1:.1%}")

# train: что выбирали при search_category = 0
t0_ = train_df[train_df["search_category"] == 0]
print("\ntrain, search_category=0:", len(t0_), "строк; категории выбранных:", t0_["item_category_id"].value_counts().to_dict())
print(t0_[["search_query", "item_title_raw", "item_category_id"]].head(10).to_string())

# --- 2) выполняются ли фильтры "Вид услуги X" на выбранных объявлениях ---
f = train_df["search_infm_params_text"].astype(str).str.strip()
simple = f.str.startswith("Вид услуги ") & ~f.str.contains("Тип услуги|Рейтинг|Кто оказывает", regex=True)
sub = train_df[simple]
ok = [a in b for a, b in zip(sub["search_infm_params_text"].str.strip(), sub["item_infm_params_text"].astype(str))]
print(f"\nПростые фильтры 'Вид услуги X' в train: {simple.sum()} строк; "
      f"выбранное объявление содержит ту же строку: {np.mean(ok):.1%}")

# любые непустые фильтры: какая доля токенов фильтра найдена в параметрах выбранного объявления
sub2 = train_df[f != ""].sample(20000, random_state=0)
def cover(fil, ip):
    ft, it = set(tokenize(fil)), set(tokenize(ip))
    return len(ft & it) / max(len(ft), 1)
cov = np.array([cover(a, b) for a, b in zip(sub2["search_infm_params_text"], sub2["item_infm_params_text"])])
print(f"Непустые фильтры: средняя доля токенов фильтра в параметрах объявления {cov.mean():.1%}; "
      f"ровно 100%: {(cov == 1.0).mean():.1%}")

Запросов с категорией 0: 222
Примеры текстов:
  - курсы украшения
  - обложка меню
  - база
  - специалист по настройкам окон пластиковых
  - микроволновая печь ремонт
  - вакансия репетитор онлайн математика информатика
  - 223 фз юрист
  - творческая студия для подростков
  - порошковая покраска высокая камера
  - водитель с авто
  - снять базу отдыха
  - карта т-банк 1500
  - стеклянный шар с наклейкой
  - репетитор английский
  - баня индивидуальное парение
  - ключ пожарный
  - трудоустройство яндекс такси
  - разработчик проектировщик мебели
  - тренер по восстановлению
  - тренер по силовым тренировкам

Категории объявлений в корпусе:
item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
Name: count, dtype: int64

Доля кандидатов вне категории 114 в нашем ответе: запросы cat=0: 5.8% | cat=114: 0.5%

train, search_category=0: 34 строк; категории выбра

In [54]:
import re
from text_utils import tokenize

RATING_RE = re.compile(r"Рейтинг пользователя \d звезды? и выше")

def filter_tokens(s):
    # рейтинговую фразу убираем: она не встречается в параметрах объявлений
    return set(tokenize(RATING_RE.sub(" ", str(s))))

print("Токенизирую параметры объявлений локального корпуса...")
t0 = time.time()
corp_param_sets = [set(tokenize(p)) for p in corpus_df["item_infm_params_text"].fillna("").astype(str)]
print(f"  готово за {time.time()-t0:.0f} c")

eval_filters = [filter_tokens(x) for x in eval_queries_df["search_infm_params_text"]]
qs_f = [qi for qi in range(len(eval_loc))
        if eval_loc[qi] in loc_to_idx and len(eval_filters[qi]) > 0][:1500]
print("Запросов с непустым фильтром и своей локацией:", len(qs_f))

res_hits = {"base": [], "hard": [], "mix35": [], "mix25": []}
pool_kept, shrunk = [], 0
for qi in tqdm(qs_f, desc="фильтр"):
    own = loc_to_idx[eval_loc[qi]]
    bm = local_bm25.score(eval_tokens[qi])
    ann = local_item_emb_v2 @ local_query_emb_v2[qi]
    tp = true_item_positions[qi]

    base = fuse_candidates_for_query(
        [topk_in(own, bm[own], 100), topk_in(own, ann[own], 100)], top_n=50)

    ft = eval_filters[qi]
    ok = np.array([ft <= corp_param_sets[i] for i in own])
    keep = own[ok]
    if len(keep) >= 50:
        hard = fuse_candidates_for_query(
            [topk_in(keep, bm[keep], 100), topk_in(keep, ann[keep], 100)], top_n=50)
        shrunk += len(keep) < len(own)
    else:
        keep, hard = own, base              # мало объявлений под фильтром -- не применяем

    res_hits["base"].append(tp in set(base))
    res_hits["hard"].append(tp in set(hard))
    res_hits["mix35"].append(tp in set(mix(hard, base, 35)))
    res_hits["mix25"].append(tp in set(mix(hard, base, 25)))
    pool_kept.append(tp in set(keep))

print(f"\nФильтр реально сузил пул у {shrunk} запросов из {len(qs_f)}")
print(f"Нужное объявление осталось в пуле после фильтра: {np.mean(pool_kept):.1%}")
for k, v in res_hits.items():
    print(f"  {k:6s} Recall@50 = {np.mean(v):.4f}")

Токенизирую параметры объявлений локального корпуса...
  готово за 13 c
Запросов с непустым фильтром и своей локацией: 1500


фильтр: 100%|██████████| 1500/1500 [00:59<00:00, 25.15it/s]


Фильтр реально сузил пул у 535 запросов из 1500
Нужное объявление осталось в пуле после фильтра: 93.3%
  base   Recall@50 = 0.8887
  hard   Recall@50 = 0.8893
  mix35  Recall@50 = 0.8907
  mix25  Recall@50 = 0.8900


In [55]:
# ---- 1) вклад ретриверов внутри локации ----
qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
names = ["bm25_only", "bert_only",
         "rrf 1:1", "rrf bm=2,bert=1", "rrf bm=1,bert=2", "rrf 1:1 k=10", "rrf 1:1 pool=300"]
hits = {n: [] for n in names}

for qi in tqdm(qs_c, desc="ретриверы"):
    own = loc_to_idx[eval_loc[qi]]
    bm = local_bm25.score(eval_tokens[qi])[own]
    ann = local_item_emb_v2[own] @ local_query_emb_v2[qi]
    tp = true_item_positions[qi]
    b100, a100 = topk_in(own, bm, 100), topk_in(own, ann, 100)
    b300, a300 = topk_in(own, bm, 300), topk_in(own, ann, 300)

    hits["bm25_only"].append(tp in set(b100[:50]))
    hits["bert_only"].append(tp in set(a100[:50]))
    hits["rrf 1:1"].append(tp in set(weighted_rrf([(b100, 1), (a100, 1)])))
    hits["rrf bm=2,bert=1"].append(tp in set(weighted_rrf([(b100, 2), (a100, 1)])))
    hits["rrf bm=1,bert=2"].append(tp in set(weighted_rrf([(b100, 1), (a100, 2)])))
    hits["rrf 1:1 k=10"].append(tp in set(weighted_rrf([(b100, 1), (a100, 1)], k=10)))
    hits["rrf 1:1 pool=300"].append(tp in set(weighted_rrf([(b300, 1), (a300, 1)])))

print("Recall@50 внутри своей локации:")
for n in names:
    print(f"  {n:20s} {np.mean(hits[n]):.4f}")

# ---- 2) как часто в train повторяется одна и та же пара (запрос, объявление) ----
q_norm = train_df["search_query"].map(lambda s: " ".join(str(s).lower().split()))
pairs = pd.DataFrame({"q": q_norm, "iid": train_df["item_id"]})
cnt = pairs.groupby(["q", "iid"]).size()
print(f"\nУникальных пар (запрос, объявление): {len(cnt)} из {len(pairs)} строк")
print(f"Строк, чья пара встречается ≥2 раз: {(pairs.merge(cnt.rename('n').reset_index(), on=['q','iid'])['n'] >= 2).mean():.1%}")

# доля строк, где тот же текст запроса уже встречался с ЭТИМ объявлением в других строках
print(f"Запросов с ≥2 строками в train: {(q_norm.value_counts() >= 2).mean():.1%} уникальных текстов")

ретриверы: 100%|██████████| 1500/1500 [00:52<00:00, 28.83it/s]


Recall@50 внутри своей локации:
  bm25_only            0.8640
  bert_only            0.8527
  rrf 1:1              0.9007
  rrf bm=2,bert=1      0.8820
  rrf bm=1,bert=2      0.8793
  rrf 1:1 k=10         0.9000
  rrf 1:1 pool=300     0.9013

Уникальных пар (запрос, объявление): 445199 из 497673 строк
Строк, чья пара встречается ≥2 раз: 17.0%
Запросов с ≥2 строками в train: 40.5% уникальных текстов


In [56]:
print("Токенизирую локальный корпус (~40 c)...")
corp_tokens = tokenize_all(corpus_item_texts)

def stem(tokens, n):
    return [t[:n] for t in tokens]

stem_idx = {}
for n in [4, 5, 6]:
    t0 = time.time()
    stem_idx[n] = BM25Index().fit([stem(t, n) for t in corp_tokens])
    print(f"  индекс по основам длины {n}: {time.time()-t0:.0f} c")

qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
names = ["bm25+bert (база)"]
for n in [4, 5, 6]:
    names += [f"stem{n} only", f"bm25+bert+stem{n}"]
hits = {k: [] for k in names}

for qi in tqdm(qs_c, desc="стемминг"):
    own = loc_to_idx[eval_loc[qi]]
    bm = local_bm25.score(eval_tokens[qi])[own]
    ann = local_item_emb_v2[own] @ local_query_emb_v2[qi]
    tp = true_item_positions[qi]
    b100, a100 = topk_in(own, bm, 100), topk_in(own, ann, 100)
    hits["bm25+bert (база)"].append(tp in set(weighted_rrf([(b100, 1), (a100, 1)])))

    for n in [4, 5, 6]:
        sb = stem_idx[n].score(stem(eval_tokens[qi], n))[own]
        s100 = topk_in(own, sb, 100)
        hits[f"stem{n} only"].append(tp in set(s100[:50]))
        hits[f"bm25+bert+stem{n}"].append(tp in set(weighted_rrf([(b100, 1), (a100, 1), (s100, 1)])))

print("\nRecall@50 внутри своей локации:")
for k in names:
    print(f"  {k:22s} {np.mean(hits[k]):.4f}")

Токенизирую локальный корпус (~40 c)...
  индекс по основам длины 4: 17 c
  индекс по основам длины 5: 14 c
  индекс по основам длины 6: 142 c


стемминг: 100%|██████████| 1500/1500 [03:39<00:00,  6.82it/s]


Recall@50 внутри своей локации:
  bm25+bert (база)       0.9007
  stem4 only             0.8800
  bm25+bert+stem4        0.9013
  stem5 only             0.8853
  bm25+bert+stem5        0.8973
  stem6 only             0.8813
  bm25+bert+stem6        0.8927


In [57]:
qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
names = ["bm25+bert (база)", "stem4+bert", "stem5+bert", "stem4+stem5+bert"]
hits = {k: [] for k in names}

for qi in tqdm(qs_c, desc="замена BM25"):
    own = loc_to_idx[eval_loc[qi]]
    bm = local_bm25.score(eval_tokens[qi])[own]
    ann = local_item_emb_v2[own] @ local_query_emb_v2[qi]
    tp = true_item_positions[qi]
    b100, a100 = topk_in(own, bm, 100), topk_in(own, ann, 100)
    s4 = topk_in(own, stem_idx[4].score(stem(eval_tokens[qi], 4))[own], 100)
    s5 = topk_in(own, stem_idx[5].score(stem(eval_tokens[qi], 5))[own], 100)

    hits["bm25+bert (база)"].append(tp in set(weighted_rrf([(b100, 1), (a100, 1)])))
    hits["stem4+bert"].append(tp in set(weighted_rrf([(s4, 1), (a100, 1)])))
    hits["stem5+bert"].append(tp in set(weighted_rrf([(s5, 1), (a100, 1)])))
    hits["stem4+stem5+bert"].append(tp in set(weighted_rrf([(s4, 0.5), (s5, 0.5), (a100, 1)])))

print("Recall@50 внутри своей локации:")
for k in names:
    print(f"  {k:20s} {np.mean(hits[k]):.4f}")

замена BM25: 100%|██████████| 1500/1500 [02:24<00:00, 10.37it/s]

Recall@50 внутри своей локации:
  bm25+bert (база)     0.9007
  stem4+bert           0.9027
  stem5+bert           0.9040
  stem4+stem5+bert     0.9067


In [58]:
def norm(s):
    return " ".join(str(s).lower().split())

qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1000]

# BERT-списки считаем один раз
ann_cache = {}
for qi in qs_c:
    own = loc_to_idx[eval_loc[qi]]
    ann_cache[qi] = topk_in(own, local_item_emb_v2[own] @ local_query_emb_v2[qi], 100)

# ---------- A: параметры BM25 (индекс по основам длины 5, вместе с BERT) ----------
stem5_docs = [stem(t, 5) for t in corp_tokens]
q5 = [stem(t, 5) for t in eval_tokens]
print("A: параметры BM25 (stem5 + bert)")
for k1, b in [(1.5, 0.75), (1.2, 0.75), (0.9, 0.4), (1.5, 0.5), (1.5, 0.9), (2.0, 0.75)]:
    idx = BM25Index(k1=k1, b=b).fit(stem5_docs)
    h = []
    for qi in qs_c:
        own = loc_to_idx[eval_loc[qi]]
        s = topk_in(own, idx.score(q5[qi])[own], 100)
        h.append(true_item_positions[qi] in set(weighted_rrf([(s, 1), (ann_cache[qi], 1)])))
    print(f"  k1={k1}, b={b}: {np.mean(h):.4f}")

# ---------- B: дедупликация копий объявлений ----------
title_key = [norm(t) for t in corpus_df["item_title_raw"].fillna("")]
text_key = [norm(t) + "|" + norm(d)[:100] for t, d in
            zip(corpus_df["item_title_raw"].fillna(""), corpus_df["item_description_raw"].fillna(""))]

def dedupe(lst, keys, n=50):
    seen, out = set(), []
    for i in lst:
        if keys[i] in seen:
            continue
        seen.add(keys[i]); out.append(i)
        if len(out) >= n:
            break
    return out

names = ["без дедупликации", "по заголовку", "по заголовку+описанию"]
hits = {k: [] for k in names}
dropped_true = {"по заголовку": 0, "по заголовку+описанию": 0}
for qi in tqdm(qs_c, desc="дедупликация"):
    own = loc_to_idx[eval_loc[qi]]
    s4 = topk_in(own, stem_idx[4].score(stem(eval_tokens[qi], 4))[own], 200)
    s5 = topk_in(own, stem_idx[5].score(stem(eval_tokens[qi], 5))[own], 200)
    a = topk_in(own, local_item_emb_v2[own] @ local_query_emb_v2[qi], 200)
    wide = weighted_rrf([(s4, 0.5), (s5, 0.5), (a, 1)], top_n=200)
    tp = true_item_positions[qi]
    hits["без дедупликации"].append(tp in set(wide[:50]))
    for name, keys in [("по заголовку", title_key), ("по заголовку+описанию", text_key)]:
        res = dedupe(wide, keys, 50)
        hits[name].append(tp in set(res))
        if tp in set(wide[:50]) and tp not in set(res):
            dropped_true[name] += 1

print("\nB: дедупликация (stem4+stem5+bert)")
for k in names:
    print(f"  {k:24s} {np.mean(hits[k]):.4f}")
print("Потеряно нужных объявлений из-за дедупликации:", dropped_true)

A: параметры BM25 (stem5 + bert)
  k1=1.5, b=0.75: 0.9080
  k1=1.2, b=0.75: 0.9080
  k1=0.9, b=0.4: 0.9070
  k1=1.5, b=0.5: 0.9080
  k1=1.5, b=0.9: 0.9080
  k1=2.0, b=0.75: 0.9100


дедупликация: 100%|██████████| 1000/1000 [01:03<00:00, 15.77it/s]


B: дедупликация (stem4+stem5+bert)
  без дедупликации         0.9110
  по заголовку             0.7470
  по заголовку+описанию    0.7870
Потеряно нужных объявлений из-за дедупликации: {'по заголовку': 174, 'по заголовку+описанию': 134}


In [59]:
# ---------- часть A: вместо радиуса в км берём N ближайших к центру локации объявлений ----------
dist_cache = {}
def dist_to_center(qi):
    if qi not in dist_cache:
        c_lat, c_lon = q_centers.loc[eval_loc[qi]]
        d = haversine_km(c_lat, c_lon, corp_lat, corp_lon)
        dist_cache[qi] = np.where(np.isnan(d), np.inf, d)
    return dist_cache[qi]

def nearest_candidates(qi, n_pool):
    d = dist_to_center(qi)
    idxs = np.argsort(d)[:n_pool]
    bm, ann = cache[qi]                      # скоры по всему корпусу, посчитаны в ячейке 20
    return fuse_candidates_for_query(
        [topk_in(idxs, bm[idxs], 100), topk_in(idxs, ann[idxs], 100)], top_n=50)

truth_miss = [true_item_positions[qi] for qi in miss]
print("Запросы без своей локации (локально):", len(miss))
preds = []
for qi in miss:
    d = dist_to_center(qi)
    idxs = np.where(d <= 100)[0]
    if len(idxs) < 200:
        idxs = np.argsort(d)[:200]
    bm, ann = cache[qi]
    preds.append(fuse_candidates_for_query(
        [topk_in(idxs, bm[idxs], 100), topk_in(idxs, ann[idxs], 100)], top_n=50))
print(build_recall_report("фиксированный радиус 100 км (как в answer_v2)", recall_at_k(preds, truth_miss)))
for N in [300, 600, 1000, 1500, 2500, 4000]:
    preds = [nearest_candidates(qi, N) for qi in miss]
    print(build_recall_report(f"{N} ближайших объявлений", recall_at_k(preds, truth_miss)))

# ---------- часть B: расстояние как третий список в RRF для запросов со своей локацией ----------
qs_c = [qi for qi in range(len(eval_loc))
        if eval_loc[qi] in loc_to_idx and eval_loc[qi] in q_centers.index][:1000]
weights = [0.0, 0.3, 0.6, 1.0]
hits = {w: [] for w in weights}
for qi in tqdm(qs_c, desc="расстояние"):
    d = dist_to_center(qi)
    wide = np.where(d <= 30)[0]              # своя локация + ближайшие соседи
    if len(wide) < 200:
        wide = np.argsort(d)[:200]
    bm = local_bm25.score(eval_tokens[qi])
    ann = local_item_emb_v2 @ local_query_emb_v2[qi]
    b300, a300 = topk_in(wide, bm[wide], 300), topk_in(wide, ann[wide], 300)
    pool = np.array(list(dict.fromkeys(b300 + a300)))
    dist_list = list(pool[np.argsort(d[pool])][:100])     # пул, отсортированный по близости
    tp = true_item_positions[qi]
    for w in weights:
        lists = [(b300[:100], 1.0), (a300[:100], 1.0)]
        if w > 0:
            lists.append((dist_list, w))
        hits[w].append(tp in set(weighted_rrf(lists)))
print("\nRecall@50, запросы со своей локацией (пул: радиус 30 км):")
for w in weights:
    print(f"  вес расстояния {w}: {np.mean(hits[w]):.4f}")

Запросы без своей локации (локально): 334
[фиксированный радиус 100 км (как в answer_v2)] локальный Recall@50 = 0.7156
[300 ближайших объявлений] локальный Recall@50 = 0.1317
[600 ближайших объявлений] локальный Recall@50 = 0.2275
[1000 ближайших объявлений] локальный Recall@50 = 0.2844
[1500 ближайших объявлений] локальный Recall@50 = 0.3473
[2500 ближайших объявлений] локальный Recall@50 = 0.4491
[4000 ближайших объявлений] локальный Recall@50 = 0.5090


расстояние: 100%|██████████| 1000/1000 [00:44<00:00, 22.72it/s]


Recall@50, запросы со своей локацией (пул: радиус 30 км):
  вес расстояния 0.0: 0.9090
  вес расстояния 0.3: 0.9120
  вес расстояния 0.6: 0.9100
  вес расстояния 1.0: 0.9000


In [60]:
import pickle

perm = np.random.default_rng(RANDOM_STATE).permutation(len(train_df))
assert (train_df.iloc[perm[:3000]]["search_query"].to_numpy()
        == eval_queries_df["search_query"].to_numpy()).all(), "порядок сплита не совпал"
free_idx = perm[150_000:]
N_TRAIN2 = 40_000
sel = np.random.default_rng(1).choice(free_idx, size=N_TRAIN2, replace=False)
tr2 = train_df.iloc[sel].reset_index(drop=True)

tr2_q_texts = build_query_texts(tr2)
tr2_i_texts = build_item_texts(tr2)
tr2_tokens = tokenize_all(tr2_q_texts)
tr2_loc = tr2["search_location_id"].to_numpy()
tr2_iid = tr2["item_id"].astype(str).to_numpy()
bench_id_to_idx = {iid: i for i, iid in enumerate(item_ids)}
all_bench = np.arange(len(item_texts))

N_HARD = 4
rng_m = np.random.default_rng(2)
hard2 = []
for k in tqdm(range(len(tr2_tokens)), desc="негативы внутри локации"):
    scores = bm25.score(tr2_tokens[k])
    idxs = loc_index.get(tr2_loc[k])
    if idxs is None or len(idxs) < 40:
        idxs = all_bench
    pos = bench_id_to_idx.get(tr2_iid[k], -1)
    top = [i for i in topk_in(idxs, scores[idxs], 31) if i != pos][:30]
    hard2.append([int(x) for x in rng_m.choice(top, size=N_HARD, replace=False)])

pickle.dump((sel, hard2), open("hard2.pkl", "wb"))
print("готово, сохранено в hard2.pkl")

негативы внутри локации: 100%|██████████| 40000/40000 [50:09<00:00, 13.29it/s]     

готово, сохранено в hard2.pkl


In [61]:
import torch

cfg2 = TwoTowerConfig(model_name="cointegrated/rubert-tiny2", projection_dim=128, max_length=64,
                      temperature=0.05, n_hard_negatives=N_HARD, batch_size=32,
                      epochs=1, lr=2e-5, device="cpu")
model2 = TwoTowerModel(cfg2)
EPOCHS2 = 5
for ep in range(EPOCHS2):
    t0 = time.time()
    model2 = train_two_tower(model2, query_texts=tr2_q_texts, item_texts_for_positive=tr2_i_texts,
                             all_item_texts=item_texts, hard_negative_indices=hard2, config=cfg2)
    torch.save({"backbone": model2.backbone.state_dict(),
                "projection": model2.projection.state_dict()}, f"two_tower_v2_ep{ep+1}.pt")
    print(f"эпоха {ep+1}/{EPOCHS2} за {(time.time()-t0)/60:.1f} мин, чекпоинт сохранён")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 12195.32it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[epoch 1/1] средний loss = 1.8374
эпоха 1/5 за 13.2 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 1.4198
эпоха 2/5 за 12.6 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 1.2396
эпоха 3/5 за 13.6 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 1.1117
эпоха 4/5 за 12.7 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 0.9979
эпоха 5/5 за 12.9 мин, чекпоинт сохранён


In [62]:
loc_item_emb_new = model2.encode(corpus_item_texts, batch_size=128)
loc_query_emb_new = model2.encode(eval_query_texts, batch_size=128)

leaked = np.isin(perm[:3000], train_sample_idx)
print(f"eval-запросов, попавших в обучение старой модели: {leaked.mean():.1%}")

qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
qs_clean = [qi for qi in qs_c if not leaked[qi]]
b_cache = {qi: topk_in(loc_to_idx[eval_loc[qi]],
                       local_bm25.score(eval_tokens[qi])[loc_to_idx[eval_loc[qi]]], 100) for qi in tqdm(qs_c)}

def run(item_emb, query_emb, qs):
    hb, hh = [], []
    for qi in qs:
        own = loc_to_idx[eval_loc[qi]]
        a100 = topk_in(own, item_emb[own] @ query_emb[qi], 100)
        tp = true_item_positions[qi]
        hb.append(tp in set(a100[:50]))
        hh.append(tp in set(weighted_rrf([(b_cache[qi], 1), (a100, 1)])))
    return np.mean(hb), np.mean(hh)

print(f"\n{'':32s} BERT only | BM25+BERT")
for name, ie, qe, qs in [
    ("старая, все запросы", local_item_emb_v2, local_query_emb_v2, qs_c),
    ("старая, без утёкших", local_item_emb_v2, local_query_emb_v2, qs_clean),
    ("новая, без утёкших", loc_item_emb_new, loc_query_emb_new, qs_clean),
]:
    b, h = run(ie, qe, qs)
    print(f"  {name:30s} {b:.4f}    | {h:.4f}")

Encoding: 100%|██████████| 24/24 [00:01<00:00, 18.61it/s]


eval-запросов, попавших в обучение старой модели: 10.6%


100%|██████████| 1500/1500 [00:55<00:00, 26.80it/s]



                                 BERT only | BM25+BERT
  старая, все запросы            0.8527    | 0.9007
  старая, без утёкших            0.8497    | 0.9030
  новая, без утёкших             0.8934    | 0.9112


In [63]:
qs_c = [qi for qi in range(len(eval_loc)) if eval_loc[qi] in loc_to_idx][:1500]
qs_clean = [qi for qi in qs_c if not leaked[qi]]

names = ["bm25 + new", "stem4+stem5 + new", "bm25 + new + old",
         "stem4+stem5 + new + old", "stem4+stem5 + 2*new"]
hits = {k: [] for k in names}
for qi in tqdm(qs_clean, desc="составы"):
    own = loc_to_idx[eval_loc[qi]]
    tp = true_item_positions[qi]
    b = topk_in(own, local_bm25.score(eval_tokens[qi])[own], 100)
    s4 = topk_in(own, stem_idx[4].score(stem(eval_tokens[qi], 4))[own], 100)
    s5 = topk_in(own, stem_idx[5].score(stem(eval_tokens[qi], 5))[own], 100)
    new = topk_in(own, loc_item_emb_new[own] @ loc_query_emb_new[qi], 100)
    old = topk_in(own, local_item_emb_v2[own] @ local_query_emb_v2[qi], 100)
    lex = [(s4, 0.5), (s5, 0.5)]

    hits["bm25 + new"].append(tp in set(weighted_rrf([(b, 1), (new, 1)])))
    hits["stem4+stem5 + new"].append(tp in set(weighted_rrf(lex + [(new, 1)])))
    hits["bm25 + new + old"].append(tp in set(weighted_rrf([(b, 1), (new, 1), (old, 0.5)])))
    hits["stem4+stem5 + new + old"].append(tp in set(weighted_rrf(lex + [(new, 1), (old, 0.5)])))
    hits["stem4+stem5 + 2*new"].append(tp in set(weighted_rrf(lex + [(new, 2)])))

print(f"Запросов в оценке: {len(qs_clean)}")
for k in names:
    print(f"  {k:26s} {np.mean(hits[k]):.4f}")

составы: 100%|██████████| 1351/1351 [02:08<00:00, 10.54it/s]

Запросов в оценке: 1351
  bm25 + new                 0.9112
  stem4+stem5 + new          0.9119
  bm25 + new + old           0.9119
  stem4+stem5 + new + old    0.9119
  stem4+stem5 + 2*new        0.9030


In [72]:
USE_STEMS = False
USE_OLD = False
BONUS_NEW = 1.0

item_emb_new = model2.encode(item_texts, batch_size=128)
query_emb_new = model2.encode(query_texts, batch_size=128)
np.save("item_embeddings_v3model.npy", item_emb_new)
np.save("query_embeddings_v3model.npy", query_emb_new)

if USE_STEMS:
    print("Индексы по основам для корпуса бенчмарка...")
    bench_stem = {n: BM25Index().fit([stem(t, n) for t in item_tokens]) for n in (4, 5)}

def sources(qi, idxs):
    ann = topk_in(idxs, (item_emb_new[idxs] @ query_emb_new[qi]), 100)
    out = [(ann, BONUS_NEW)]
    if USE_OLD:
        out.append((topk_in(idxs, item_emb_v2[idxs] @ query_emb_v2[qi], 100), 0.5))
    if USE_STEMS:
        for n in (4, 5):
            sc = bench_stem[n].score(stem(bench_tokens[qi], n))
            out.append((topk_in(idxs, sc[idxs], 100), 0.5))
    else:
        out.append((topk_in(idxs, bm25.score(bench_tokens[qi])[idxs], 100), 1.0))
    return out

final_v3 = []
n_loc, n_radius, n_glob = 0, 0, 0
for qi in tqdm(range(len(query_ids)), desc="Финальные кандидаты"):
    own = loc_index.get(bench_q_loc[qi])
    if own is not None and len(own) >= 1:
        res = weighted_rrf(sources(qi, own))
        n_loc += 1
    else:
        res = []
        if bench_q_loc[qi] in q_centers.index:
            c_lat, c_lon = q_centers.loc[bench_q_loc[qi]]
            d = haversine_km(c_lat, c_lon, bench_lat, bench_lon)
            d = np.where(np.isnan(d), np.inf, d)
            idxs = np.where(d <= 100)[0]
            if len(idxs) < 200:
                idxs = np.argsort(d)[:200]
            res = weighted_rrf(sources(qi, idxs))
            n_radius += 1
    if len(res) < 50:
        n_glob += 1
        seen = set(res)
        for i in weighted_rrf(sources(qi, all_idx), top_n=100):
            if len(res) >= 50:
                break
            if i not in seen:
                res.append(i); seen.add(i)
    final_v3.append(res[:50])

print(f"Своя локация: {n_loc}, радиус: {n_radius}, добор глобально: {n_glob}")
answer_v4 = build_answer_dataframe(query_ids, final_v3, item_ids)
problems = validate_answer_df(answer_v4, query_ids, item_ids, max_items_per_row=50)
print("Проблемы формата:", problems if problems else "нет")
write_answer_csv(answer_v4, "answer_v4.csv")
print("answer_v4.csv записан:", answer_v4.shape)

Финальные кандидаты: 100%|██████████| 2452/2452 [01:48<00:00, 22.62it/s]


Своя локация: 2025, радиус: 427, добор глобально: 8
Проблемы формата: нет
answer_v4.csv записан: (2452, 2)


In [65]:
used = set(sel.tolist())
rest = np.array([i for i in free_idx if i not in used])
N_NEW = 60_000
sel3 = np.random.default_rng(3).choice(rest, size=N_NEW, replace=False)
tr3 = train_df.iloc[sel3].reset_index(drop=True)

tr3_q_texts = build_query_texts(tr3)
tr3_i_texts = build_item_texts(tr3)
tr3_tokens = tokenize_all(tr3_q_texts)
tr3_loc = tr3["search_location_id"].to_numpy()
tr3_iid = tr3["item_id"].astype(str).to_numpy()

rng_m = np.random.default_rng(4)
hard3 = []
for k in tqdm(range(len(tr3_tokens)), desc="негативы внутри локации"):
    scores = bm25.score(tr3_tokens[k])
    idxs = loc_index.get(tr3_loc[k])
    if idxs is None or len(idxs) < 40:
        idxs = all_bench
    pos = bench_id_to_idx.get(tr3_iid[k], -1)
    top = [i for i in topk_in(idxs, scores[idxs], 31) if i != pos][:30]
    hard3.append([int(x) for x in rng_m.choice(top, size=N_HARD, replace=False)])

pickle.dump((sel3, hard3), open("hard3.pkl", "wb"))
print("готово, hard3.pkl сохранён")

негативы внутри локации: 100%|██████████| 60000/60000 [44:19<00:00, 22.56it/s]


готово, hard3.pkl сохранён


In [66]:
cfg3 = TwoTowerConfig(model_name="cointegrated/rubert-tiny2", projection_dim=128, max_length=64,
                      temperature=0.05, n_hard_negatives=N_HARD, batch_size=32,
                      epochs=1, lr=1e-5, device="cpu")
model2.config = cfg3
for ep in range(3):
    t0 = time.time()
    model2 = train_two_tower(model2, query_texts=tr3_q_texts, item_texts_for_positive=tr3_i_texts,
                             all_item_texts=item_texts, hard_negative_indices=hard3, config=cfg3)
    torch.save({"backbone": model2.backbone.state_dict(),
                "projection": model2.projection.state_dict()}, f"two_tower_v3_ep{ep+1}.pt")
    print(f"эпоха {ep+1}/3 за {(time.time()-t0)/60:.1f} мин, чекпоинт сохранён")

[epoch 1/1] средний loss = 1.2873
эпоха 1/3 за 23.8 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 1.1976
эпоха 2/3 за 29.8 мин, чекпоинт сохранён
[epoch 1/1] средний loss = 1.1206
эпоха 3/3 за 36.9 мин, чекпоинт сохранён


In [67]:
loc_item_emb_v3 = model2.encode(corpus_item_texts, batch_size=128)
loc_query_emb_v3 = model2.encode(eval_query_texts, batch_size=128)
print(f"\n{'':22s} BERT only | BM25+BERT")
for name, ie, qe in [("v2 (0.8144)", loc_item_emb_new, loc_query_emb_new),
                     ("v3 (+60k пар)", loc_item_emb_v3, loc_query_emb_v3)]:
    b, h = run(ie, qe, qs_clean)
    print(f"  {name:20s} {b:.4f}    | {h:.4f}")

Encoding: 100%|██████████| 24/24 [00:01<00:00, 19.62it/s]



                       BERT only | BM25+BERT
  v2 (0.8144)          0.8934    | 0.9112
  v3 (+60k пар)        0.9001    | 0.9171


In [71]:
for name in ["item_emb_new", "query_emb_new", "final_v3", "answer_v4"]:
    globals().pop(name, None)

chk = model2.encode(corpus_item_texts[:64], batch_size=64, show_progress=False)

USE_STEMS, USE_OLD, BONUS_NEW = False, False, 1.0

item_emb_new = model2.encode(item_texts, batch_size=128)
query_emb_new = model2.encode(query_texts, batch_size=128)
np.save("item_embeddings_v3model.npy", item_emb_new)
np.save("query_embeddings_v3model.npy", query_emb_new)

def sources(qi, idxs):
    ann = topk_in(idxs, (item_emb_new[idxs] @ query_emb_new[qi]), 100)
    bm = topk_in(idxs, bm25.score(bench_tokens[qi])[idxs], 100)
    return [(ann, BONUS_NEW), (bm, 1.0)]

final_v4 = []
n_loc = n_radius = n_glob = 0
for qi in tqdm(range(len(query_ids)), desc="Финальные кандидаты v4"):
    own = loc_index.get(bench_q_loc[qi])
    res = []
    if own is not None and len(own) >= 1:
        res = weighted_rrf(sources(qi, own)); n_loc += 1
    elif bench_q_loc[qi] in q_centers.index:
        c_lat, c_lon = q_centers.loc[bench_q_loc[qi]]
        d = haversine_km(c_lat, c_lon, bench_lat, bench_lon)
        d = np.where(np.isnan(d), np.inf, d)
        idxs = np.where(d <= 100)[0]
        if len(idxs) < 200:
            idxs = np.argsort(d)[:200]
        res = weighted_rrf(sources(qi, idxs)); n_radius += 1
    if len(res) < 50:
        n_glob += 1
        seen = set(res)
        for i in weighted_rrf(sources(qi, all_idx), top_n=100):
            if len(res) >= 50:
                break
            if i not in seen:
                res.append(i); seen.add(i)
    final_v4.append(res[:50])

print(f"Своя локация: {n_loc}, радиус: {n_radius}, добор глобально: {n_glob}")

answer_v4 = build_answer_dataframe(query_ids, final_v4, item_ids)
problems = validate_answer_df(answer_v4, query_ids, item_ids, max_items_per_row=50)
write_answer_csv(answer_v4, "answer_v4.csv")
changed = (answer_v4["answer"].values != answer_v3["answer"].values).mean()
print(f"answer_v4.csv записан: {answer_v4.shape}")

model2 совпадает с v3: True


Encoding: 100%|██████████| 20/20 [00:00<00:00, 21.08it/s]


эмбеддинги сохранены: (189212, 128) (2452, 128)


Финальные кандидаты v4: 100%|██████████| 2452/2452 [01:51<00:00, 21.91it/s]


Своя локация: 2025, радиус: 427, добор глобально: 8
Проблемы формата: нет
answer_v4.csv записан: (2452, 2); отличается от answer_v3 у 100.0% запросов
